# Step 0 Task 8 — Live Verifier Spike

**Goal:** Run the offline verifier spike unattended on Kaggle CPU/GPU:
1. Verify the 20-sign evaluation list against the GISLR 250 vocabulary.
2. Calibrate `EYE_TO_SHOULDER_RATIO` and `EYE_TO_SHOULDER_Y_OFFSET` from visible pose frames.
3. Build k-medoids reference clusters and per-phase ranges from training signers only.
4. Evaluate candidate judges on held-out signers (correct pass, minimal pair false-green, perturbations).
5. Export `reports/step0/verifier.md`.

In [ ]:
# 1. Environment & Imports
from pathlib import Path

import numpy as np
import pandas as pd

from learnsign.data.split import signer_split
from learnsign.verify.calibrate import calibrate_eye_ratios
from learnsign.verify.evaluate import evaluate_judge
from learnsign.verify.judges import CombinedJudge, ModelJudge, RangesJudge, SpecJudge
from learnsign.verify.run_spike import (
    _load,
    _raw_pose,
    build_examples,
    format_report,
)
from learnsign.verify.signs import load_step0_signs, validate_glosses

print("Task 8 verifier modules loaded successfully.")

In [ ]:
# 2. Dataset Paths
DATA_DIR = Path("/kaggle/input/asl-signs")
TRAIN_CSV = DATA_DIR / "train.csv"
REPORT_OUT = Path("reports/step0/verifier.md")
CHECKPOINT_PATH = Path("models/quick_baseline.pt")

if not TRAIN_CSV.exists():
    print("Note: /kaggle/input/asl-signs not found. Using local or mock data path.")

In [ ]:
# 3. Verify Signs and Signer-Independent Split
signs = load_step0_signs()
n_signs = len(signs["signs"])
n_pairs = len(signs["minimal_pairs"])
print(f"Loaded {n_signs} evaluation signs including {n_pairs} minimal pairs.")

if TRAIN_CSV.exists():
    train = pd.read_csv(TRAIN_CSV)
    missing = validate_glosses(train["sign"].unique())
    assert not missing, f"Missing signs: {missing}"
    train_df, held_df = signer_split(train, n_val_signers=4, seed=42)
    n_signers = held_df["participant_id"].nunique()
    print(f"Train: {len(train_df)}, held-out: {len(held_df)} ({n_signers} signers).")
else:
    print("Kaggle dataset pending. Ready for unattended run.")

In [ ]:
# 4. Calibrate Eye Fallback Proportions
if TRAIN_CSV.exists():
    sample_rows = train_df.sample(n=min(300, len(train_df)), random_state=42)
    poses = np.concatenate([_raw_pose(DATA_DIR, p) for p in sample_rows["path"]])
    calibration = calibrate_eye_ratios(poses)
    print("Eye-to-shoulder ratio:", calibration["eye_to_shoulder_ratio"])
    print("Eye-to-shoulder y-offset:", calibration["eye_to_shoulder_y_offset"])
else:
    calibration = {
        "eye_to_shoulder_ratio": {"median": 6.0, "p05": 5.2, "p95": 6.8},
        "eye_to_shoulder_y_offset": {"median": 0.5, "p05": 0.42, "p95": 0.58},
        "frames_used": 0,
    }
    print("Default baseline calibration parameters retained pending Kaggle run.")

In [ ]:
# 5. Build Candidates and Evaluate
notes = [
    "References: training signers only, 6 clips per sign; scored attempts: held-out signers.",
    "Frame rate assumed 30 fps (GISLR parquets have no timestamps).",
]

if TRAIN_CSV.exists():
    references = {
        g: [_load(DATA_DIR, p) for p in train_df[train_df["sign"] == g].head(6)["path"]]
        for g in signs["signs"]
    }
    examples = build_examples(DATA_DIR, held_df, signs)
    ranges = RangesJudge(references, k=3)
    candidates = {"(a) data ranges": ranges, "(b) LLM specs": SpecJudge()}
    if CHECKPOINT_PATH.exists():
        from learnsign.verify.judges import load_quick_baseline_predictor
        gloss_names = sorted(train["sign"].unique())
        model = ModelJudge(load_quick_baseline_predictor(CHECKPOINT_PATH, gloss_names))
        candidates["(c) trained model"] = model
        candidates["(d) ranges + model"] = CombinedJudge(ranges, model)
    else:
        notes.append("No checkpoint given: candidates (c) and (d) not run.")
    results = {name: evaluate_judge(j, examples) for name, j in candidates.items()}
    report_md = format_report(results, calibration, notes)
    REPORT_OUT.parent.mkdir(parents=True, exist_ok=True)
    REPORT_OUT.write_text(report_md, encoding="utf-8")
    print(report_md)
else:
    print("Awaiting Kaggle run for live GISLR held-out numbers.")